In [10]:
from langchain_community.vectorstores import FAISS
from langchain_cohere import CohereEmbeddings
from langchain_classic.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document

In [11]:
from dotenv import load_dotenv
import os

In [26]:
docs = [

Document(page_content="Langchain helps to buid LLM Applications."),
Document(page_content="Pinecone is a Vector database for semantic search"),
Document(page_content="The Eiffel tower is located in Paris"),
Document(page_content="Langchain can be used for Agentic AI Applications"),
Document(page_content="Langchain has many types of retrievers")
]

In [27]:
# Dense retriver
embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
)

In [28]:
dense_vectors = FAISS.from_documents(docs, embedding=embeddings)
dense_retrievers= dense_vectors.as_retriever()

In [34]:
# Sparse Retriver (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=2

Combine with Ensemble retriever

In [35]:
hybrid_retriver =EnsembleRetriever(
    retrievers =[dense_retrievers , sparse_retriever],
    weights=[0.7,0.3]
)

In [36]:
hybrid_retriver

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'CohereEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A8AC6F7080>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001A8AC6F76B0>, k=2)], weights=[0.7, 0.3])

In [37]:
response = hybrid_retriver.invoke("How can i build an application using LLM?")

In [38]:
for i , doc in enumerate(response):
    print(f"\n Document{i+1} : \n {doc.page_content}")



 Document1 : 
 Langchain can be used for Agentic AI Applications

 Document2 : 
 Langchain has many types of retrievers

 Document3 : 
 Langchain helps to buid LLM Applications.

 Document4 : 
 Pinecone is a Vector database for semantic search


RAG Pipeline with Hybrid retiever

In [39]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [40]:
prompt = PromptTemplate.from_template("""
    Answser the question based on the context below.
                                      
Context :{context}
Question :{input}                                

""")

In [41]:
from langchain_groq import ChatGroq
groq_llm = ChatGroq(model="llama-3.1-8b-instant")

Create stuff document chain

In [42]:
document_chain = create_stuff_documents_chain(groq_llm , prompt=prompt)

create full RAG Chain = create Retrieval chain

In [44]:
rag_chain = create_retrieval_chain(retriever=hybrid_retriver, combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'CohereEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A8AC6F7080>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001A8AC6F76B0>, k=2)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answser the question based on the context below.\n\nContext :{context}\nQuestion :{input}                                \n\n')
 

In [45]:
query ={"input":"How can build an app using LLM ?"}

response = rag_chain.invoke(query)

In [47]:
print("Answer : \n " , response['answer'])

print("\n Source Documents")

for i, doc in enumerate(response["context"]):
    print(f"\nDOc {i+1}:{doc.page_content}")

Answer : 
  Based on the context, it seems that you want to build an application using a Large Language Model (LLM). 

To build an app using an LLM, you can use Langchain, which helps to build LLM applications. Here's a general outline of the steps you can follow:

1. **Choose a Langchain Retriever**: Langchain has various types of retrievers that can be used to fetch relevant data from a database or the internet. You can choose the one that best suits your needs, such as a Pinecone Vector database retriever for semantic search.

2. **Connect to a Database or Data Source**: If you're using a retriever like Pinecone, you'll need to connect to a vector database or another data source that contains the relevant information for your application.

3. **Build an LLM Model**: You can use Langchain's capabilities to build and train an LLM model that can understand and respond to user input.

4. **Integrate with a UI or API**: Once you have your LLM model built, you can integrate it with a user